In [1]:
import os
import copy
import time
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ── Step 1: Make sure model is on CPU (dynamic quant is CPU-only) ─────────────

In [2]:
def quantize_model(model: nn.Module) -> nn.Module:
    """
    Apply PyTorch Dynamic Quantization.
    Targets: nn.LSTM and nn.Linear layers → weights go INT8.
    Activations are quantized on-the-fly during inference.
    """
    model_cpu = copy.deepcopy(model).cpu()
    model_cpu.eval()
 
    quantized = torch.quantization.quantize_dynamic(
        model_cpu,
        qconfig_spec={nn.LSTM, nn.Linear},   # layers to quantize
        dtype=torch.qint8                     # INT8 weights
    )
    return quantized

# ── Step 2: Model size utility ────────────────────────────────────────────────

In [ ]:
C:\IIoT-Predictive-Maintenance\processed\gru_lstm_FD001.pt

In [3]:
def get_model_size_mb(model: nn.Module, path: str = "_tmp_model.pt") -> float:
    """Save model to disk and return file size in MB."""
    torch.save(model.state_dict(), path)
    size_mb = os.path.getsize(path) / (1024 ** 2)
    os.remove(path)
    return size_mb

# ── Step 3: Inference speed benchmark ────────────────────────────────────────

In [4]:
def benchmark_inference(model: nn.Module, X_test: np.ndarray,
                        batch_size: int = 64, runs: int = 5) -> float:
    """Returns average inference time in milliseconds over `runs` passes."""
    model.eval()
    dataset = torch.utils.data.TensorDataset(torch.tensor(X_test))
    loader  = torch.utils.data.DataLoader(dataset, batch_size=batch_size)
 
    times = []
    with torch.no_grad():
        for _ in range(runs):
            t0 = time.perf_counter()
            for (batch,) in loader:
                _ = model(batch.cpu())
            times.append((time.perf_counter() - t0) * 1000)
 
    return float(np.mean(times))

# ── Step 4: Predict helper ────────────────────────────────────────────────────

In [5]:
def predict_numpy(model: nn.Module, X_test: np.ndarray, batch_size: int = 64):
    """Run inference and return predictions as a numpy array."""
    model.eval()
    preds = []
    dataset = torch.utils.data.TensorDataset(torch.tensor(X_test))
    loader  = torch.utils.data.DataLoader(dataset, batch_size=batch_size)
    with torch.no_grad():
        for (batch,) in loader:
            preds.append(model(batch.cpu()).numpy())
    return np.concatenate(preds).flatten()

# ── Step 5: NASA scoring function ─────────────────────────────────────────────

In [6]:
def score_fn(y_true, y_pred):
    diff = y_pred - y_true
    return float(np.sum(np.where(diff < 0,
                                 np.exp(-diff / 13) - 1,
                                 np.exp(diff / 10) - 1)))

# ── Main comparison routine ───────────────────────────────────────────────────

In [7]:
def run_quantization_study(model: nn.Module,
                           X_test: np.ndarray,
                           y_test: np.ndarray,
                           dataset_name: str = "FD001"):
    """
    Full PTQ study:
      - Quantize the trained FP32 model
      - Compare size, speed, RMSE, MAE, NASA score
      - Print a clear pass/fail verdict
    """
 
    print(f"\n{'='*58}")
    print(f"  Dynamic Quantization Study — {dataset_name}")
    print(f"{'='*58}")
 
    # ── FP32 baseline ──
    fp32_model = copy.deepcopy(model).cpu().eval()
    fp32_size  = get_model_size_mb(fp32_model)
    fp32_time  = benchmark_inference(fp32_model, X_test)
    fp32_preds = predict_numpy(fp32_model, X_test)
 
    fp32_rmse  = np.sqrt(mean_squared_error(y_test, fp32_preds))
    fp32_mae   = mean_absolute_error(y_test, fp32_preds)
    fp32_score = score_fn(y_test, fp32_preds)
 
    # ── INT8 quantized ──
    print("Applying dynamic quantization …")
    int8_model = quantize_model(model)
    int8_size  = get_model_size_mb(int8_model)
    int8_time  = benchmark_inference(int8_model, X_test)
    int8_preds = predict_numpy(int8_model, X_test)
 
    int8_rmse  = np.sqrt(mean_squared_error(y_test, int8_preds))
    int8_mae   = mean_absolute_error(y_test, int8_preds)
    int8_score = score_fn(y_test, int8_preds)
 
    # ── Deltas ──
    size_reduction   = (1 - int8_size  / fp32_size)  * 100
    speedup          = fp32_time / int8_time
    rmse_delta_pct   = (int8_rmse  - fp32_rmse)  / fp32_rmse  * 100
    mae_delta_pct    = (int8_mae   - fp32_mae)   / fp32_mae   * 100
    score_delta_pct  = (int8_score - fp32_score) / abs(fp32_score) * 100
 
    # ── Report ──
    print(f"\n{'─'*58}")
    print(f"  {'Metric':<22} {'FP32':>10} {'INT8':>10} {'Δ':>10}")
    print(f"{'─'*58}")
    print(f"  {'Model size (MB)':<22} {fp32_size:>10.3f} {int8_size:>10.3f} {-size_reduction:>+9.1f}%")
    print(f"  {'Inference (ms)':<22} {fp32_time:>10.1f} {int8_time:>10.1f} {speedup:>9.2f}x")
    print(f"  {'RMSE':<22} {fp32_rmse:>10.4f} {int8_rmse:>10.4f} {rmse_delta_pct:>+9.2f}%")
    print(f"  {'MAE':<22} {fp32_mae:>10.4f} {int8_mae:>10.4f} {mae_delta_pct:>+9.2f}%")
    print(f"  {'NASA Score':<22} {fp32_score:>10.1f} {int8_score:>10.1f} {score_delta_pct:>+9.2f}%")
    print(f"{'─'*58}")
 
    # ── Verdict ──
    RMSE_THRESHOLD = 5.0   # % degradation considered acceptable
    if rmse_delta_pct <= RMSE_THRESHOLD:
        verdict = "✅  PTQ SUCCESSFUL — accuracy loss is within acceptable range."
        recommendation = (
            "You can deploy the INT8 model. Save it with:\n"
            "  torch.save(int8_model, 'lstm_int8_FD001.pt')\n"
            "  (Use torch.save on the whole model, not just state_dict,\n"
            "   because quantized models need their custom ops bundled.)"
        )
    else:
        verdict = "⚠️  PTQ FAILED — accuracy degraded too much."
        recommendation = (
            "Upgrade to Quantization-Aware Training (QAT):\n"
            "  model.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')\n"
            "  torch.quantization.prepare_qat(model, inplace=True)\n"
            "  # ← retrain for ~10% of original epochs\n"
            "  torch.quantization.convert(model.eval(), inplace=True)"
        )
 
    print(f"\n  {verdict}")
    print(f"\n  Next step:\n  {recommendation}")
    print(f"{'='*58}\n")
 
    return int8_model, {
        "fp32": {"size_mb": fp32_size, "rmse": fp32_rmse,
                 "mae": fp32_mae, "nasa_score": fp32_score},
        "int8": {"size_mb": int8_size, "rmse": int8_rmse,
                 "mae": int8_mae, "nasa_score": int8_score},
        "size_reduction_pct": size_reduction,
        "speedup": speedup,
    }

In [30]:
'''import sys
sys.path.insert(0, r"C:\\IIoT-Predictive-Maintenance")
from GRU_LSTM import LSTMModel, load_data, add_rul, normalize, make_test_sequences, SELECTED_SENSORS, SEQ_LEN, HIDDEN_SIZE, NUM_LAYERS, DROPOUT, MAX_RUL
'''

'import sys\nsys.path.insert(0, r"C:\\IIoT-Predictive-Maintenance")\nfrom GRU_LSTM import LSTMModel, load_data, add_rul, normalize, make_test_sequences, SELECTED_SENSORS, SEQ_LEN, HIDDEN_SIZE, NUM_LAYERS, DROPOUT, MAX_RUL\n'

In [ ]:
if __name__ == "__main__":
    """
    Standalone mode: loads a saved FP32 model and test data to run the study.
    Adjust paths and imports to match your project layout.
    """
    import sys
    sys.path.insert(0, ".")
    from processed import (
        gru_lstm_FD001, load_data, add_rul, normalize,
        make_test_sequences, SELECTED_SENSORS, SEQ_LEN,
        HIDDEN_SIZE, NUM_LAYERS, DROPOUT, MAX_RUL
    )
 
    DATASET  = "FD001"
    DATA_DIR = "C:\\IIoT-Predictive-Maintenance\\dataset\\train_FD001.txt"           # folder with train_FD001.txt etc.
 
    # Rebuild model and load weights
    model = LSTMModel(len(SELECTED_SENSORS), HIDDEN_SIZE, NUM_LAYERS, DROPOUT)
    model.load_state_dict(torch.load(f"gru_lstm_{DATASET}.pt", map_location="cpu"))
 
    # Rebuild test sequences
    train_df, test_df, rul_df = load_data(DATA_DIR, DATASET)
    train_df = add_rul(train_df)
    train_df, test_df, _ = normalize(train_df, test_df, SELECTED_SENSORS)
    X_test = make_test_sequences(test_df, SELECTED_SENSORS, SEQ_LEN)
    y_test = np.clip(rul_df["RUL"].values.astype(np.float32), 0, MAX_RUL)
 
    int8_model, results = run_quantization_study(model, X_test, y_test, DATASET)

ImportError: cannot import name 'LSTMModel' from 'processed' (unknown location)